# Example: Single spin Part 3: Single qubit gate optimisation using GRAPE

#### 1. Generate a PWC pulse shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from cthree.signal.Envelopes import GaussEnvelope
from cthree.signal.SimpleGenerator import PWCGenerator

First, let's generate a piecewise constant (PWC) pulse envelope for the Gaussian pulse

In [ ]:
t_final = 20e-9
tlist = np.linspace(0, t_final, 21)
tone = GaussEnvelope()
tone.t_final.setValue(t_final)
gen = PWCGenerator(envelopes=[tone], tlist=tlist)

In [ ]:
params = gen.getParameters()

In [ ]:
params[1] = params[1]

In [ ]:
ts = np.linspace(0, t_final, 1001)

plt.plot(ts / 1e-9, tone.computeOutput(ts) / 1e6, label="Smooth curve")
plt.plot(
    ts / 1e-9, np.real(gen.generateSignal(ts)) / 1e6, ls="--", label="Inphase"
)
plt.plot(
    ts / 1e-9,
    np.imag(gen.generateSignal(ts)) / 1e6,
    ls="--",
    label="Quadrature",
)

plt.xlabel("Time [in ns]")
plt.ylabel("Amplitude [in MHz]")
plt.legend()

#### 2. Define Hamiltonian in the rotating frame of drive

Next, we setup the qubit system we want to control. We define the Hamiltonian in the rotating frame of drive such that the pulse oscillates slowly to apply GRAPE gradients.

The Hamiltonain in the rotating frame of the drive is given by - 
$$ H(t) = \big(\omega_q - \omega_d\big) b^\dagger b -\frac{\alpha}{2} (b^\dagger)^2 b^2 + (\epsilon(t) b + \epsilon(t)^* b)$$

In [ ]:
from cthree.Quantity import Quantity
from cthree.model.ClosedModel import ClosedModel
from cthree.model.RotatingFrameDrive import RotatingFrameDrive
from cthree.model.Transmon import Transmon


FREQ = 7.86e9 * 2 * np.pi
DIMS = 3
ANHARM = -264e6 * 2 * np.pi
offset = -50e6 * 2 * np.pi

drive_freq = FREQ + offset
qubit_freq = FREQ - drive_freq

Drive = RotatingFrameDrive(gen)
transmon = Transmon(
    frequency=Quantity(
        qubit_freq,
        0.8 * qubit_freq,
        1.2 * qubit_freq,
        unit="Hz",
        name="Frequency",
    ),
    anharmonicity=Quantity(
        ANHARM, 1.2 * ANHARM, 0.8 * ANHARM, unit="Hz", name="Anharmonicity"
    ),
    drives=[Drive],
    dimension=DIMS,
)

model = ClosedModel(transmon)

In [ ]:
from cthree.measurement.StateTransferFidelity import StateTransferFidelityGRAPE
from cthree.propagation.ScipyExpmGRAPE import ScipyExpmGRAPE


prop = ScipyExpmGRAPE(model, res=100e9)

init = np.array([[1.0], [0], [0]])  # |0>
target = np.array([[0.0], [1], [0]])  # |1>

prop.setInitialState(init)
prop.setTargetState(target)

zeroone = StateTransferFidelityGRAPE(
    propagation=prop,
    initialState=init,
    targetState=target,
    times=tlist,
)

In [ ]:
def plotStates():
    """Plot the states."""
    ts = np.linspace(0, t_final, 1001)
    states = prop.propagate(ts)
    sig = gen.generateSignal(ts)

    fig, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts / 1e-9, np.real(sig), label="I")
    ax[0].plot(ts / 1e-9, np.imag(sig), label="Q")
    ax[0].legend(loc=1)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts / 1e-9, np.abs(states)[:, :, 0] ** 2)
    ax[1].set_ylabel("Population")
    ax[-1].set_xlabel("Time [ns]")
    return fig, ax


plotStates()

As expected, we get a partial transfer and a low fidelity.

In [ ]:
zeroone.measure()

We define an optimizer and link our fidelity measure as a goal function and the parameters of the cosine tone and optimise just amplitude and frequency, as in the state transfer example.

In [ ]:
from cthree.OptimisationMap import OptimisationMap
from cthree.optimisers.ScipyOptimiserGradient import ScipyOptimiserGradient


optmap = OptimisationMap()
optmap.add(gen, params)
opt = ScipyOptimiserGradient(zeroone, optimisables=optmap)

In [ ]:
opt.optimise()

In [ ]:
ts = np.linspace(0, t_final, 1001)

plt.plot(
    ts / 1e-9, np.real(gen.generateSignal(ts)) / 1e6, ls="--", label="Inphase"
)
plt.plot(
    ts / 1e-9,
    np.imag(gen.generateSignal(ts)) / 1e6,
    ls="--",
    label="Quadrature",
)

plt.xlabel("Time [in ns]")
plt.ylabel("Amplitude [in MHz]")
plt.legend()

In [ ]:
plotStates()